In [ ]:
# Load in merged data frame 
# Runs merge_station_events 

import pandas as pd
import numpy as np
import os
import math

from src.cleaning.clean_stations import clean_station_data
from src.cleaning.merge_station_events import merge_station_events
from src.cleaning.merged_tornado_indicator import create_tornado_indicator

## Data Access and Setup
columns_to_drop = ['NAME',
                    'SOURCE',
                    'REPORT_TYPE',
                    'CALL_SIGN',
                    'QUALITY_CONTROL',
                    'CALL_SIGN.1',
                    'QUALITY_CONTROL.1',
                    'REPORT_TYPE.1',
                    'SOURCE.1',
                    'AB1',
                    'AD1',
                    'AE1',
                    'AG1',
                    'AH1',
                    'AH2',
                    'AH3',
                    'AH4',
                    'AH5',
                    'AH6',
                    'AI1',
                    'AI2',
                    'AI3',
                    'AI4',
                    'AI5',
                    'AI6',
                    'AK1',
                    'AM1',
                    'AN1',
                    'AT1',
                    'AT2',
                    'AT3',
                    'AT4',
                    'AT5',
                    'AT6',
                    'AT7',
                    'AT8',
                    'AU1',
                    'AU2',
                    'AU3',
                    'AU4',
                    'AU5',
                    'AW1',
                    'AW2',
                    'AW3',
                    'AW4',
                    'AW5',
                    'AW6',
                    'AW7',
                    'AX1',
                    'AX2',
                    'AX3',
                    'AX4',
                    'AX5',
                    'AX6',
                    'ED1',
                    'EQD',
                    'GD1',
                    'GD2',
                    'GD3',
                    'GD4',
                    'GE1',
                    'GF1',
                    'IA1',
                    'KC1',
                    'KC2',
                    'KD1',
                    'KD2',
                    'KE1',
                    'MH1',
                    'MK1',
                    'MV1',
                    'MW1',
                    'MW2',
                    'MW3',
                    'MW4',
                    'MW5',
                    'OD1',
                    'OE1',
                    'OE2',
                    'OE3',
                    'REM',
                    'SA1',
                    'UA1',
                    'UG1',
                    'WA1',
                    'AA1',
                    'AA2',
                    'AA3',
                    'AA4',
                    'AJ1',
                    'AL1',
                    'GA2',
                    'GA3',
                    'GA4',
                    'GA5',
                    'GA6',
                    'GJ1',
                    'GK1',
                    'GP1',
                    'GQ1',
                    'GR1',
                    'HL1',
                    'KA1',
                    'KA2',
                    'KA3',
                    'KA4',
                    'KB1',
                    'KB2',
                    'KB3',
                    'KG1',
                    'KG2',
                    'MD1',
                    'MF1',
                    'MG1',
                    'OC1',
                    'RH1',
                    'RH2',
                    'RH3'
                    ]
### Reasons to get rid of 
# 'NAME': already have an identifier column 'STATION'.
# 'SOURCE': this is just the source or sources used to create sample.
# 'REPORT_TYPE': denotes the type of geophysical surface observation.
# 'CALL_SIGN': call letters assigned to a weather station. We already have an identifier.
# 'QUALITY_CONTROl': For predicting tornadoes, this might not be super useful.
# though it may be good to keep in mind if so desired. One can erase all V01 
# entries (no quality control).
# 'CALL_SIGN.1': see CALL_SIGN.
# 'QUALITY_CONTROL.1' : see QUALITY_CONTROL.
# 'REPORT_TYPE.1': see REPORT_TYPE.
# 'SOURCE.1': see SOURCE.
# 'AB1': Liquid Precipitation Monthly total-- too long of a time scale.
# 'AD1: Liquid Precipitation Greatest Amount in 24 Hours, For the month -- too long of a time scale.
# 'AE1': Number of Days with Specific Amounts for Each Month -- Can be obtained through AA1-AA4
# 'AG1': 'Precipitation Estimated Observation -- not sure how this is different from AA1-AA4
# 'AH1'-- AH6' : Liquid Precipitation Maximum Short Duration, For The Month -- too long of a time scale.
# 'AI1 -- AI6' : Identical to 'AH1'--'AH6'
# 'AK1 Greatest Snow Depth on Ground for the Month
# 'AM1':
# 'AN1':
# 'AT1--AT8': Data leakage
# 'AU1--AU5': Data leakage 
# 'AW1--AW7': Data leakage
# 'AX1--AX6': Data leakage
# 'ED1': Runway Visibility
# 'EQD':
# 'GD1--GD4': Similar to GA1-GA6
# 'GE1': Similar to GA1-GA6 (may include later)
# 'GF1': Similar to GA1-GA6 (may include later)
# 'IA1': 
# 'KC1--KD2': too long of a time scale
# 'KE1': Extreme Temperatures, Number of Days Exceeding Criteria, For the Month -- too long of a time scale
# 'MH1': Atmospheric Pressure Observation for the month -- too long of a time scale.
# 'MK1' : See 'MH1'
# 'MV1 : Present Weather in Vicinity Observation -- Potential Data Leakage
# 'MW1--MW5' : Present Weather Observation -- Potential Data Leakage 
# 'OE1--OE3': Already have WND
# 'REM' : These are remarks
# 'SA1': Sea Surface Temperature 
# 'UA1'--'UG1' Marine Data?
# 'WA1'-- Platform Ice accretion ###


# SPLIT TUPLES IN PARTICULAR COLUMNS

# These are ordered in the way their tuples are ordered

mapping = {

'CIG': ['CIG- Sky Condition Observation- Ceiling Height Dimension',
       'CIG- Sky Condition Observation- Ceiling Quality Code',
       'CIG- Sky Condition Observation- Ceiling Determination Code',
       'CIG- Sky Condition Observation- Cavok Code'],

'DEW':['DEW- Air Temperature Observation- Dew Point Temperature',
       'DEW- Air Temperature Observation- Dew Point Quality Code'],

'GA1':['GA1- Sky Cover Layer- Coverage Code',
       'GA1- Sky Cover Layer- Coverage Quality Code',
       'GA1- Sky Cover Layer- Base Height Dimensions',
       'GA1- Sky Cover Layer- Base Height Quality Code',
       'GA1- Sky Cover Layer- Cloud Type Code',
       'GA1- Sky Cover Layer- Cloud Type Quality Code'],

'MA1':['MA1-Atmospheric Pressure Observation- Altimeter Setting Rate',
       'MA1-Atmospheric Pressure Observation- Altimeter Quality Code',
       'MA1-Atmospheric Pressure Observation- Station Pressure Rate',
       'MA1-Atmospheric Pressure Observation- Station Pressure Quality Code'],


'SLP':['SLP- Atmospheric Pressure Observation- Sea Level Pressure',
       'SLP- Atmospheric Pressure Observation- Sea Level Pressure Quality Code'],

'TMP':['TMP- Air Temperature Observation- Air Temperature',
       'TMP- Air Temperature Observation- Air Temperature Quality Code'],

'VIS': ['VIS- Visibility Observation- Distance Dimension',
       'VIS- Visibility Observation- Distance Quality Code',
       'VIS- Visibility Observation- Variability Code',
       'VIS- Visibility Observation- Quality Variability Code'],

'WND':['WND- Wind Observation- Direction Angle',
       'WND- Wind Observation- Direction Quality Code',
       'WND- Wind Observation- Type Code',
       'WND- Wind Observation- Speed Rate',
       'WND- Wind Observation- Speed Quality Code'],
}



data = create_tornado_indicator(drop_cols=columns_to_drop,split_tuples=True, mapping=mapping, tuple_sep=',', time_window=1,drop_originals=True,val_radius=50)